# 08 — PCA Brain Scores: Status & Pace

Extracts two subject-level brain scores via PC1:
- **Brain status** (PC1 of cross-sectional Z-scores at baseline)
- **Brain pace** (PC1 of longitudinal delta-Z scores V1→V2)

**Inputs:**
- `Z_all_timepoints_healthy.csv` / `Z_all_timepoints.csv` (from `05_longitudinal_predict.ipynb` or `06_longitudinal_predict_healthy.ipynb`)
- `DeltaZ_V1_V2_healthy.csv` … `DeltaZ_V3_V4_healthy.csv` (from `06_longitudinal_predict_healthy.ipynb`)
- For patient transfer: `Z_all_timepoints.csv` and `DeltaZ_V1_V2.csv` (from `05_longitudinal_predict.ipynb`)

**Outputs:** PC1 scores & loadings per visit-pair/visit, `finalscore.csv`

**`FIT_ON_HEALTHY_ONLY = True` (recommended):** PCA is fit on healthy subjects only; patients are projected onto healthy loadings — the normative approach.  
**`FIT_ON_HEALTHY_ONLY = False`:** PCA is fit on all subjects together.

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
# Directory containing Z_all_timepoints*.csv and DeltaZ_*.csv files
LONGITUDINAL_DIR = 'outputs/longitudinal'
HEALTHY_DIR      = 'outputs/longitudinal_healthy'

# All PCA outputs are written here
OUTPUT_DIR       = 'outputs/pca_brain_scores'

# Final merged score file
FINALSCORE_CSV   = 'outputs/finalscore.csv'

# True  → fit PCA on healthy subjects, then project patients (normative approach)
# False → fit PCA on all subjects together
FIT_ON_HEALTHY_ONLY = True

# Which visit pair to use for brain pace and which visit for brain status
PACE_PAIR    = 'V1_V2'   # longitudinal delta-Z pair
STATUS_VISIT = 'V1'      # cross-sectional visit (ses-00A baseline)

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'PCA_Longitudinal'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'PCA_CrossSectional'), exist_ok=True)

## ROI categories (lobe labels)

In [ ]:
VOL_CATEGORIES = {
    # Subcortical
    'mr_y_smri__vol__aseg__ag__lh_sum': 'Subcortical_Left',  'mr_y_smri__vol__aseg__ag__rh_sum': 'Subcortical_Right',
    'mr_y_smri__vol__aseg__cd__lh_sum': 'Subcortical_Left',  'mr_y_smri__vol__aseg__cd__rh_sum': 'Subcortical_Right',
    'mr_y_smri__vol__aseg__hc__lh_sum': 'Subcortical_Left',  'mr_y_smri__vol__aseg__hc__rh_sum': 'Subcortical_Right',
    'mr_y_smri__vol__aseg__pl__lh_sum': 'Subcortical_Left',  'mr_y_smri__vol__aseg__pl__rh_sum': 'Subcortical_Right',
    'mr_y_smri__vol__aseg__pt__lh_sum': 'Subcortical_Left',  'mr_y_smri__vol__aseg__pt__rh_sum': 'Subcortical_Right',
    'mr_y_smri__vol__aseg__th__lh_sum': 'Subcortical_Left',  'mr_y_smri__vol__aseg__th__rh_sum': 'Subcortical_Right',
    'mr_y_smri__vol__aseg__ab__lh_sum': 'Subcortical_Left',  'mr_y_smri__vol__aseg__ab__rh_sum': 'Subcortical_Right',
    # Temporal
    'mr_y_smri__vol__dsk__ins__lh_sum':    'Temporal_Left',  'mr_y_smri__vol__dsk__ins__rh_sum':    'Temporal_Right',
    'mr_y_smri__vol__dsk__stmp__lh_sum':   'Temporal_Left',  'mr_y_smri__vol__dsk__stmp__rh_sum':   'Temporal_Right',
    'mr_y_smri__vol__dsk__ttmp__lh_sum':   'Temporal_Left',  'mr_y_smri__vol__dsk__ttmp__rh_sum':   'Temporal_Right',
    'mr_y_smri__vol__dsk__bstmps__lh_sum': 'Temporal_Left',  'mr_y_smri__vol__dsk__bstmps__rh_sum': 'Temporal_Right',
    'mr_y_smri__vol__dsk__ptmp__lh_sum':   'Temporal_Left',  'mr_y_smri__vol__dsk__ptmp__rh_sum':   'Temporal_Right',
    'mr_y_smri__vol__dsk__er__lh_sum':     'Temporal_Left',  'mr_y_smri__vol__dsk__er__rh_sum':     'Temporal_Right',
    'mr_y_smri__vol__dsk__mtmp__lh_sum':   'Temporal_Left',  'mr_y_smri__vol__dsk__mtmp__rh_sum':   'Temporal_Right',
    'mr_y_smri__vol__dsk__ff__lh_sum':     'Temporal_Left',  'mr_y_smri__vol__dsk__ff__rh_sum':     'Temporal_Right',
    'mr_y_smri__vol__dsk__itmp__lh_sum':   'Temporal_Left',  'mr_y_smri__vol__dsk__itmp__rh_sum':   'Temporal_Right',
    'mr_y_smri__vol__dsk__ph__lh_sum':     'Temporal_Left',  'mr_y_smri__vol__dsk__ph__rh_sum':     'Temporal_Right',
    # Frontal
    'mr_y_smri__vol__dsk__pfrt__lh_sum':   'Frontal_Left',   'mr_y_smri__vol__dsk__pfrt__rh_sum':   'Frontal_Right',
    'mr_y_smri__vol__dsk__prcn__lh_sum':   'Frontal_Left',   'mr_y_smri__vol__dsk__prcn__rh_sum':   'Frontal_Right',
    'mr_y_smri__vol__dsk__pactr__lh_sum':  'Frontal_Left',   'mr_y_smri__vol__dsk__pactr__rh_sum':  'Frontal_Right',
    'mr_y_smri__vol__dsk__mobfrt__lh_sum': 'Frontal_Left',   'mr_y_smri__vol__dsk__mobfrt__rh_sum': 'Frontal_Right',
    'mr_y_smri__vol__dsk__lobfrt__lh_sum': 'Frontal_Left',   'mr_y_smri__vol__dsk__lobfrt__rh_sum': 'Frontal_Right',
    'mr_y_smri__vol__dsk__pop__lh_sum':    'Frontal_Left',   'mr_y_smri__vol__dsk__pop__rh_sum':    'Frontal_Right',
    'mr_y_smri__vol__dsk__pob__lh_sum':    'Frontal_Left',   'mr_y_smri__vol__dsk__pob__rh_sum':    'Frontal_Right',
    'mr_y_smri__vol__dsk__ptg__lh_sum':    'Frontal_Left',   'mr_y_smri__vol__dsk__ptg__rh_sum':    'Frontal_Right',
    'mr_y_smri__vol__dsk__cmfrt__lh_sum':  'Frontal_Left',   'mr_y_smri__vol__dsk__cmfrt__rh_sum':  'Frontal_Right',
    'mr_y_smri__vol__dsk__cac__lh_sum':    'Frontal_Left',   'mr_y_smri__vol__dsk__cac__rh_sum':    'Frontal_Right',
    'mr_y_smri__vol__dsk__prctr__lh_sum':  'Frontal_Left',   'mr_y_smri__vol__dsk__prctr__rh_sum':  'Frontal_Right',
    'mr_y_smri__vol__dsk__sfrt__lh_sum':   'Frontal_Left',   'mr_y_smri__vol__dsk__sfrt__rh_sum':   'Frontal_Right',
    'mr_y_smri__vol__dsk__rmfrt__lh_sum':  'Frontal_Left',   'mr_y_smri__vol__dsk__rmfrt__rh_sum':  'Frontal_Right',
    'mr_y_smri__vol__dsk__rac__lh_sum':    'Frontal_Left',   'mr_y_smri__vol__dsk__rac__rh_sum':    'Frontal_Right',
    # Parietal
    'mr_y_smri__vol__dsk__iprt__lh_sum':   'Parietal_Left',  'mr_y_smri__vol__dsk__iprt__rh_sum':   'Parietal_Right',
    'mr_y_smri__vol__dsk__pcg__lh_sum':    'Parietal_Left',  'mr_y_smri__vol__dsk__pcg__rh_sum':    'Parietal_Right',
    'mr_y_smri__vol__dsk__poctr__lh_sum':  'Parietal_Left',  'mr_y_smri__vol__dsk__poctr__rh_sum':  'Parietal_Right',
    'mr_y_smri__vol__dsk__ic__lh_sum':     'Parietal_Left',  'mr_y_smri__vol__dsk__ic__rh_sum':     'Parietal_Right',
    'mr_y_smri__vol__dsk__sm__lh_sum':     'Parietal_Left',  'mr_y_smri__vol__dsk__sm__rh_sum':     'Parietal_Right',
    'mr_y_smri__vol__dsk__sprt__lh_sum':   'Parietal_Left',  'mr_y_smri__vol__dsk__sprt__rh_sum':   'Parietal_Right',
    # Occipital
    'mr_y_smri__vol__dsk__pcc__lh_sum':    'Occipital_Left', 'mr_y_smri__vol__dsk__pcc__rh_sum':    'Occipital_Right',
    'mr_y_smri__vol__dsk__locc__lh_sum':   'Occipital_Left', 'mr_y_smri__vol__dsk__locc__rh_sum':   'Occipital_Right',
    'mr_y_smri__vol__dsk__lg__lh_sum':     'Occipital_Left', 'mr_y_smri__vol__dsk__lg__rh_sum':     'Occipital_Right',
    'mr_y_smri__vol__dsk__cn__lh_sum':     'Occipital_Left', 'mr_y_smri__vol__dsk__cn__rh_sum':     'Occipital_Right',
}

## Shared PCA helper

In [ ]:
def run_pca(df, label, out_subdir):
    """Fit PC1 on df, save scores and loadings. Returns (scores_df, loadings_df)."""
    valid_cols = [c for c in df.columns if c in VOL_CATEGORIES]
    df_num = df[valid_cols].dropna()
    if df_num.empty:
        print(f'  SKIP {label}: no valid rows after dropna')
        return None, None

    X = StandardScaler().fit_transform(df_num)
    pca = PCA(n_components=1)
    scores = pca.fit_transform(X)
    print(f'  {label}: {len(df_num)} subjects | PC1 variance = {pca.explained_variance_ratio_[0]:.2%}')

    scores_df = pd.DataFrame(scores, columns=['PC1'], index=df_num.index)
    if 'group' in df.columns:
        scores_df = scores_df.join(df[['group']], how='left')

    loadings_df = pd.DataFrame(
        pca.components_.T, index=valid_cols, columns=['PC1_Loading']
    )
    loadings_df['Region'] = loadings_df.index.map(VOL_CATEGORIES)
    loadings_df = loadings_df.sort_values('PC1_Loading', ascending=False)

    scores_df.to_csv(os.path.join(out_subdir, f'PC1_Scores_{label}.csv'))
    loadings_df.to_csv(os.path.join(out_subdir, f'PC1_Loadings_{label}.csv'))
    return scores_df, loadings_df


def project_onto_loadings(df, loadings_df):
    """Project subjects in df onto pre-fitted loadings (dot product). Returns PC1 series."""
    rois = loadings_df.index.tolist()
    common = [r for r in rois if r in df.columns]
    weights = loadings_df.loc[common, 'PC1_Loading'].values
    return pd.Series(df[common].values @ weights, index=df.index, name='PC1')

## Step 1 — Longitudinal PCA (brain pace: delta-Z)

In [ ]:
DELTA_PAIRS = ['V1_V2', 'V1_V3', 'V1_V4', 'V2_V3', 'V2_V4', 'V3_V4']
long_out = os.path.join(OUTPUT_DIR, 'PCA_Longitudinal')

print('--- Longitudinal PCA ---')
for pair in DELTA_PAIRS:
    suffix = '_healthy' if FIT_ON_HEALTHY_ONLY else ''
    src_dir = HEALTHY_DIR if FIT_ON_HEALTHY_ONLY else LONGITUDINAL_DIR
    fpath = os.path.join(src_dir, f'DeltaZ_{pair}{suffix}.csv')
    if not os.path.exists(fpath):
        print(f'  SKIP {pair}: {fpath} not found')
        continue
    df = pd.read_csv(fpath).set_index('ID')
    label = f'{pair}_healthy' if FIT_ON_HEALTHY_ONLY else pair
    run_pca(df, label, long_out)

print('Longitudinal PCA complete.')

## Step 2 — Cross-sectional PCA (brain status: Z-scores)

In [ ]:
VISITS = {'V1': 'ses-00A', 'V2': 'ses-02A', 'V3': 'ses-04A', 'V4': 'ses-06A'}
cs_out = os.path.join(OUTPUT_DIR, 'PCA_CrossSectional')

suffix  = '_healthy' if FIT_ON_HEALTHY_ONLY else ''
src_dir = HEALTHY_DIR if FIT_ON_HEALTHY_ONLY else LONGITUDINAL_DIR
z_file  = os.path.join(src_dir, f'Z_all_timepoints{suffix}.csv')

if not os.path.exists(z_file):
    raise FileNotFoundError(f'Z-score file not found: {z_file}')

df_z = pd.read_csv(z_file)
idx_col = 'ID-wave' if 'ID-wave' in df_z.columns else df_z.columns[0]
df_z = df_z.rename(columns={idx_col: 'ID-wave'}).set_index('ID-wave')
print(f'Loaded {z_file}: {len(df_z)} rows')

print('--- Cross-sectional PCA ---')
for v_label, session_id in VISITS.items():
    df_visit = df_z[df_z.index.str.contains(session_id)].copy()
    df_visit.index = df_visit.index.str.split('_ses').str[0]
    df_visit.index.name = 'ID'
    v_suffix = f'{v_label}_healthy' if FIT_ON_HEALTHY_ONLY else v_label
    run_pca(df_visit, v_suffix, cs_out)

print('Cross-sectional PCA complete.')

## Step 3 — Project patients onto healthy loadings

Only runs when `FIT_ON_HEALTHY_ONLY = True`.  
Patients are NOT re-fitted; their data is projected onto the PC axes derived from healthy subjects.

In [ ]:
pat_pace_scores = None
pat_status_scores = None

if FIT_ON_HEALTHY_ONLY:
    print('--- Patient projection onto healthy loadings ---')

    # ── Brain pace (longitudinal) ───────────────────────────────────────────
    delta_all_path     = os.path.join(LONGITUDINAL_DIR, f'DeltaZ_{PACE_PAIR}.csv')
    loadings_pace_path = os.path.join(long_out, f'PC1_Loadings_{PACE_PAIR}_healthy.csv')

    if os.path.exists(delta_all_path) and os.path.exists(loadings_pace_path):
        df_all_delta   = pd.read_csv(delta_all_path).set_index('ID')
        patients_delta = df_all_delta[df_all_delta['group'] == 'Patient'].copy()
        loadings_pace  = pd.read_csv(loadings_pace_path, index_col=0)

        pac_scores     = project_onto_loadings(patients_delta, loadings_pace)
        pat_pace_scores = pd.DataFrame({'PC1': pac_scores, 'group': patients_delta['group']})
        print(f'  Pace: projected {len(pat_pace_scores)} patients onto {PACE_PAIR} loadings')
    else:
        print(f'  SKIP pace projection: missing {delta_all_path} or {loadings_pace_path}')

    # ── Brain status (cross-sectional) ─────────────────────────────────────
    # Load Z_all_timepoints.csv (all subjects), filter to baseline, then
    # keep PATIENTS ONLY by intersecting with patient IDs from the pace step.
    # Healthy scores come directly from the PCA in Step 2 — no duplication.
    z_all_path           = os.path.join(LONGITUDINAL_DIR, 'Z_all_timepoints.csv')
    loadings_status_path = os.path.join(cs_out, f'PC1_Loadings_{STATUS_VISIT}_healthy.csv')

    if os.path.exists(z_all_path) and os.path.exists(loadings_status_path):
        df_z_all = pd.read_csv(z_all_path)
        idx_col  = 'ID-wave' if 'ID-wave' in df_z_all.columns else df_z_all.columns[0]
        df_z_all = df_z_all.rename(columns={idx_col: 'ID-wave'})

        # Baseline rows only
        baseline_session = VISITS[STATUS_VISIT]
        df_base = df_z_all[df_z_all['ID-wave'].str.contains(baseline_session)].copy()
        df_base['ID'] = df_base['ID-wave'].str.split('_ses').str[0]
        df_base = df_base.set_index('ID')

        # Restrict to patients — mirrors original inner-join with pat[['group']]
        if pat_pace_scores is not None:
            df_base = df_base.loc[df_base.index.intersection(pat_pace_scores.index)].copy()

        loadings_status = pd.read_csv(loadings_status_path, index_col=0)
        stat_scores     = project_onto_loadings(df_base, loadings_status)
        pat_status_scores = pd.DataFrame({'PC1': stat_scores})
        if 'group' in df_base.columns:
            pat_status_scores['group'] = df_base['group']
        print(f'  Status: projected {len(pat_status_scores)} patients onto {STATUS_VISIT} loadings')
    else:
        print(f'  SKIP status projection: missing {z_all_path} or {loadings_status_path}')

else:
    print('FIT_ON_HEALTHY_ONLY=False — skipping patient projection step.')

## Step 4 — Merge PC1_status + PC1_pace → finalscore.csv

In [ ]:
if FIT_ON_HEALTHY_ONLY:
    # Load healthy scores
    healthy_pace_path   = os.path.join(long_out, f'PC1_Scores_{PACE_PAIR}_healthy.csv')
    healthy_status_path = os.path.join(cs_out,   f'PC1_Scores_{STATUS_VISIT}_healthy.csv')

    pace_healthy   = pd.read_csv(healthy_pace_path,   index_col='ID')
    status_healthy = pd.read_csv(healthy_status_path, index_col='ID')

    # Combine healthy + patient projected scores
    if pat_pace_scores is not None:
        pca_zdiff = pd.concat([pace_healthy[['PC1', 'group']], pat_pace_scores], axis=0)
    else:
        pca_zdiff = pace_healthy[['PC1', 'group']]

    if pat_status_scores is not None:
        pca_z = pd.concat([status_healthy[['PC1']], pat_status_scores[['PC1']]], axis=0)
    else:
        pca_z = status_healthy[['PC1']]

else:
    # Load all-subjects scores directly
    pace_path   = os.path.join(long_out, f'PC1_Scores_{PACE_PAIR}.csv')
    status_path = os.path.join(cs_out,   f'PC1_Scores_{STATUS_VISIT}.csv')

    pca_zdiff = pd.read_csv(pace_path,   index_col='ID')
    pca_z     = pd.read_csv(status_path, index_col='ID')

# Rename and merge
pca_zdiff = pca_zdiff.rename(columns={'PC1': 'PC1_pace'})
pca_z     = pca_z.rename(columns={'PC1': 'PC1_status'})

finalscore = pd.concat([pca_zdiff, pca_z[['PC1_status']]], axis=1, join='inner')
finalscore.index.name = 'ID'
finalscore.to_csv(FINALSCORE_CSV)

print(f'Saved: {FINALSCORE_CSV}  ({len(finalscore)} subjects)')
print(finalscore.head())